# M7 · Per-layer cache budgets

So far every layer got the same number of slots. But M4 showed that some layers
use a few hot experts (a small cache is enough) and others spread out (need more slots).

**Experiment** (`src/budgets.py`):
1. On the **fit** half: run LRU with equal slots and measure each layer's miss rate.
2. Keep the **same total** number of slots, but give each layer slots in proportion
   to its miss rate (at least 8, at most 64).
3. On the unseen **test** half: compare equal vs reallocated.

"No gain" is a valid result: it means the layers behave alike.

**Kaggle:** Accelerator = **None** (CPU is enough, saves GPU quota), Internet = On.
**Inputs:** Add Input → Datasets → **moe-traces** (from M2).

In [ ]:
# ---- Setup: get the latest project code (run this cell first, every session) ----
import os, sys

REPO_URL = "https://github.com/YOUR-TEAM/moe-offload.git"   # <-- change to your team's repo
# Private repo? Add a Kaggle Secret called GITHUB_TOKEN (Add-ons -> Secrets), then uncomment:
# from kaggle_secrets import UserSecretsClient
# REPO_URL = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")

if os.path.exists("/kaggle"):                      # running on Kaggle
    import glob, shutil
    CODE_DIR = "/kaggle/working/moe-offload"
    # Option A: the code was uploaded as a Kaggle dataset ("moe-offload-code") and added as an input
    uploaded = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/moe-offload-code/config.py", recursive=True)]
    if uploaded:
        shutil.copytree(uploaded[0], CODE_DIR, dirs_exist_ok=True)
        print("using code from dataset:", uploaded[0])
    # Option B: get the code from GitHub
    elif not os.path.exists(CODE_DIR):
        !git clone -q {REPO_URL} {CODE_DIR}         # first time: download the code
    else:
        !git -C {CODE_DIR} pull -q                  # later: get the newest commits
    os.chdir(CODE_DIR)
    !pip install -q -r requirements.txt
else:                                              # running on your own computer
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")                             # go to the repo root

sys.path.insert(0, os.getcwd())                    # so "import config" and "from src..." work
!git log -1 --format="Code version: %h  %s"

In [ ]:
import os
import pandas as pd
import config
from src.files import load_trace
from src.simulator import split_trace
from src.budgets import compare_uniform_vs_reallocated
from src.calibrate import load_copy_time

copy_time = load_copy_time()
all_results, rows = {}, []
for dataset in ["wikitext", "code"]:
    fit, test = split_trace(load_trace(dataset))
    for slots in [16, 32]:            # 8 slots leaves nothing to move around
        r = compare_uniform_vs_reallocated(fit, test, slots, copy_time_s=copy_time)
        all_results[(dataset, slots)] = r
        rows.append({"dataset": dataset, "slots_per_layer": slots,
                     "uniform_hit_rate": r["uniform"]["hit_rate"],
                     "reallocated_hit_rate": r["reallocated"]["hit_rate"],
                     "uniform_stall_ms": r["uniform"]["stall_ms_per_token"],
                     "reallocated_stall_ms": r["reallocated"]["stall_ms_per_token"]})
        print(dataset, slots, "-> slots per layer:", r["reallocated_slots"])

budgets = pd.DataFrame(rows)
budgets["stall_saved_%"] = 100 * (1 - budgets.reallocated_stall_ms / budgets.uniform_stall_ms)
budgets.to_csv(os.path.join(config.RESULTS_DIR, "budgets.csv"), index=False)
budgets.round(3)

In [ ]:
import matplotlib.pyplot as plt

dataset, slots = "wikitext", 16          # change to look at other cases
r = all_results[(dataset, slots)]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].bar(range(16), r["reallocated_slots"], label="reallocated")
axes[0].axhline(slots, c="red", ls="--", label="equal split")
axes[0].set(title="Slots per layer", xlabel="layer"); axes[0].legend()
axes[1].plot(r["uniform"]["hit_rate_per_layer"], marker="o", label="equal split")
axes[1].plot(r["reallocated"]["hit_rate_per_layer"], marker="o", label="reallocated")
axes[1].set(title="Hit rate per layer (test half)", xlabel="layer"); axes[1].legend()
plt.tight_layout(); plt.savefig(os.path.join(config.RESULTS_DIR, f"M7_budgets_{dataset}_{slots}.png"), dpi=130)
plt.show()

## ✅ Done when
`results/budgets.csv` exists. In `docs/M7.md`: did reallocation help, by how much, and which layers got more slots?